# Data Exploration — Revelio Labs Academic Sample

Explores all 7 CSVs and saves column-summary tables as `.tex` files to `outputs/`.

**To use in Overleaf:** add `\input{outputs/filename}` where you want the table.

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to wherever the repo sits in your Google Drive
    ROOT = Path('/content/drive/MyDrive/wage-as-signal')
except ImportError:
    IN_COLAB = False
    ROOT = Path.cwd()
    if ROOT.name == 'code':
        ROOT = ROOT.parent

RAW     = ROOT / 'data' / 'raw'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

print(f'Running {"on Colab" if IN_COLAB else "locally"}')
print(f'Root : {ROOT}')
print(f'Raw  : {RAW}')

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

In [ ]:
# ── File registry ──────────────────────────────────────────────────────────
FILES = {
    'individual_positions' : RAW / 'revelio_academic_individual' / 'academic_individual_position_academic.csv',
    'individual_users'     : RAW / 'revelio_academic_individual' / 'academic_individual_user_academic.csv',
    'individual_education' : RAW / 'revelio_academic_individual' / 'academic_individual_user_education_academic.csv',
    'individual_skills'    : RAW / 'revelio_academic_individual' / 'academic_individual_user_skill_academic.csv',
    'postings_indeed'      : RAW / 'revelio_academic_postings'   / 'academic_postings_indeed_individual_academic.csv',
    'postings_linkedin'    : RAW / 'revelio_academic_postings'   / 'academic_postings_linkedin_individual_academic.csv',
    'postings_unified'     : RAW / 'revelio_academic_postings'   / 'academic_postings_unified_individual_academic.csv',
}

In [ ]:
# ── Helper: summarise one CSV ──────────────────────────────────────────────
def summarise(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Returns (shape_df, column_df) for a CSV file."""
    df = pd.read_csv(path, low_memory=False)

    rows = []
    for col in df.columns:
        n_miss = df[col].isna().sum()
        pct    = round(n_miss / len(df) * 100, 1)

        # date range for date-like columns
        date_range = ''
        if any(k in col.lower() for k in ('date', 'start', 'end', 'year')):
            try:
                parsed = pd.to_datetime(df[col], errors='coerce')
                lo, hi = parsed.min(), parsed.max()
                if pd.notna(lo):
                    date_range = f"{lo.date()} – {hi.date()}"
            except Exception:
                pass

        rows.append({
            'Column'       : col,
            'Type'         : str(df[col].dtype),
            'Missing (\\%)': pct,
            'Date range'   : date_range,
            'Sample values': ', '.join(df[col].dropna().astype(str).unique()[:3]),
        })

    col_df   = pd.DataFrame(rows)
    shape_df = pd.DataFrame([{'File': path.name, 'Rows': len(df), 'Columns': len(df.columns)}])
    return df, shape_df, col_df

In [ ]:
# ── Run exploration ────────────────────────────────────────────────────────
results   = {}
shapes    = []

for name, path in FILES.items():
    print(f'Loading {name}...')
    df, shape_df, col_df = summarise(path)
    results[name] = {'df': df, 'columns': col_df}
    shapes.append(shape_df)

shape_summary = pd.concat(shapes, ignore_index=True)
print('\nDone.\n')
display(shape_summary)

In [ ]:
# ── Inspect each file ─────────────────────────────────────────────────────
for name, res in results.items():
    print(f'\n{"-" * 60}')
    print(f'  {name}')
    print(f'{"-" * 60}')
    display(res['columns'])
    print('\nSample rows:')
    display(res['df'].head(3))

In [ ]:
# ── Save .tex files to outputs/ ───────────────────────────────────────────
def to_tex(df: pd.DataFrame, label: str, caption: str) -> str:
    col_fmt = 'l' * len(df.columns)
    body = df.to_latex(
        index=False,
        escape=False,
        column_format=col_fmt,
    )
    # Wrap in a float with caption and label
    return (
        '\\begin{table}[htbp]\n'
        '\\centering\n'
        '\\small\n'
        f'\\caption{{{caption}}}\n'
        f'\\label{{tab:{label}}}\n'
        + body +
        '\\end{table}\n'
    )

# Overview table
tex = to_tex(shape_summary, 'data_overview', 'Revelio Labs — file overview')
(OUTPUTS / 'data_overview.tex').write_text(tex, encoding='utf-8')
print('Saved data_overview.tex')

# One codebook table per file
for name, res in results.items():
    col_df = res['columns'].drop(columns=['Sample values'])  # keep tex clean
    tex = to_tex(col_df, f'codebook_{name}', f'Codebook: {name.replace("_", " ")}')
    out_path = OUTPUTS / f'codebook_{name}.tex'
    out_path.write_text(tex, encoding='utf-8')
    print(f'Saved {out_path.name}')

print('\nAll .tex files written to outputs/')